In [1]:
import pandas as pd
import numpy as np
from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem, Descriptors, MACCSkeys
from rdkit.Chem.SaltRemover import SaltRemover
import rdkit.Chem.Fragments as Fragments

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import xgboost as xgb

# -------------------------------
# 1. Load Data and Prepare Molecules
# -------------------------------
data_df = pd.read_csv("./dw_data/Opt1_acidic_tr.csv")
smiles_col = 'OriginalSmiles'
target = 'pKa'

# Convert SMILES to RDKit Mol objects and remove salts
saltRemover = SaltRemover(defnFilename='./Salts.txt')
smiles_mols = data_df[smiles_col].astype(str).apply(Chem.MolFromSmiles)
rdkit_mols = smiles_mols.apply(saltRemover.StripMol)

# -------------------------------
# 2. Compute Continuous Descriptors & Fragment Counts (Fixed Features)
# -------------------------------
def compute_continuous_and_fragment_features(mol):
    if mol is None:
        return {}
    features = {}
    features["MolWt"] = Descriptors.MolWt(mol)
    features["MolLogP"] = Descriptors.MolLogP(mol)
    features["NumRotatableBonds"] = Descriptors.NumRotatableBonds(mol)
    features["fr_Al_OH"] = Fragments.fr_Al_OH(mol)
    features["fr_Ar_N"] = Fragments.fr_Ar_N(mol)
    return features

features_list = [compute_continuous_and_fragment_features(mol) for mol in rdkit_mols]
features_df = pd.DataFrame(features_list)  # continuous/fragment features

# -------------------------------
# 3. Precompute MACCS Keys for All Molecules (Fixed 167 bits)
# -------------------------------
def get_maccs_fingerprint(mol):
    if mol is None:
        return None
    maccs = MACCSkeys.GenMACCSKeys(mol)
    arr = np.zeros((maccs.GetNumBits(),), dtype=int)
    DataStructs.ConvertToNumpyArray(maccs, arr)
    return arr

maccs_list = [get_maccs_fingerprint(mol) for mol in rdkit_mols]

# -------------------------------
# 4. Function to Generate Morgan Fingerprints (Parameters Vary)
# -------------------------------
def get_morgan_fingerprint(mol, radius=2, nBits=1024):
    if mol is None:
        return None
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=radius, nBits=nBits)
    arr = np.zeros((nBits,), dtype=int)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

# -------------------------------
# 5. Define XGBoost GridSearch Parameters & Training Function
# -------------------------------
param_grid = {
    'n_estimators': [300],
    'max_depth': [3, 6],
    'learning_rate': [0.1, 0.2],
    'subsample': [0.8],
    'colsample_bytree': [0.8, 1.0],
    'reg_lambda': [1, 5]
}

def train_and_evaluate(X, y, random_state=11):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=random_state)
    xgbr = xgb.XGBRegressor(random_state=42, objective='reg:squarederror')
    grid_search = GridSearchCV(estimator=xgbr, param_grid=param_grid,
                               scoring='r2', cv=5, n_jobs=5, verbose=0)
    grid_search.fit(X_train, y_train)
    best_model = grid_search.best_estimator_
    y_pred_train = best_model.predict(X_train)
    y_pred_test = best_model.predict(X_test)
    metrics = {
        'best_params': str(grid_search.best_params_),
        'Train_R2': r2_score(y_train, y_pred_train),
        'Test_R2': r2_score(y_test, y_pred_test),
        'Test_MAE': mean_absolute_error(y_test, y_pred_test),
        'Test_MSE': mean_squared_error(y_test, y_pred_test),
        'Test_RMSE': np.sqrt(mean_squared_error(y_test, y_pred_test))
    }
    return metrics

# -------------------------------
# 6. Experiment 1: Morgan Fingerprints Only
# -------------------------------
radii = [1, 2, 3, 4]
sizes = [12, 64, 128, 256, 512, 1024, 2048, 4096, 8192, 16384]

morgan_results = []
for radius in radii:
    for size in sizes:
        # Compute Morgan fingerprints for all molecules with given radius and size
        morgan_fps = [get_morgan_fingerprint(mol, radius=radius, nBits=size) for mol in rdkit_mols]
        # Filter valid indices (exclude molecules where fingerprint generation failed)
        valid_indices = [i for i, fp in enumerate(morgan_fps) if fp is not None]
        if not valid_indices:
            continue
        morgan_fps = [morgan_fps[i] for i in valid_indices]
        y_morgan = data_df[target].values[valid_indices]
        X_morgan = np.vstack(morgan_fps).astype(float)
        metrics = train_and_evaluate(X_morgan, y_morgan)
        metrics.update({'radius': radius, 'fp_size': size})
        morgan_results.append(metrics)

morgan_results_df = pd.DataFrame(morgan_results)
morgan_results_df.to_csv("morgan_results.csv", index=False)
print("Morgan fingerprint experiment results saved to morgan_results.csv")

# -------------------------------
# 7. Experiment 2: MACCS Keys Only
# -------------------------------
valid_indices_maccs = [i for i, fp in enumerate(maccs_list) if fp is not None]
maccs_filtered = [maccs_list[i] for i in valid_indices_maccs]
y_maccs = data_df[target].values[valid_indices_maccs]
X_maccs = np.vstack(maccs_filtered).astype(float)
maccs_metrics = train_and_evaluate(X_maccs, y_maccs)
maccs_metrics_df = pd.DataFrame([maccs_metrics])
maccs_metrics_df.to_csv("maccs_results.csv", index=False)
print("MACCS experiment results saved to maccs_results.csv")

# -------------------------------
# 8. Experiment 3: Continuous Descriptors & Fragment Counts Only
# -------------------------------
# Assuming all molecules are valid for continuous descriptors
X_cont = features_df.values.astype(float)
y_cont = data_df[target].values
cont_metrics = train_and_evaluate(X_cont, y_cont)
cont_metrics_df = pd.DataFrame([cont_metrics])
cont_metrics_df.to_csv("continuous_results.csv", index=False)
print("Continuous descriptors/fragment counts experiment results saved to continuous_results.csv")

# -------------------------------
# 9. Experiment 4: Combined Features (Morgan + MACCS + Continuous)
# -------------------------------
combined_results = []
for radius in radii:
    for size in sizes:
        # Morgan fingerprints
        morgan_fps = [get_morgan_fingerprint(mol, radius=radius, nBits=size) for mol in rdkit_mols]
        # Filter valid indices where both Morgan and MACCS are available
        valid_indices = [i for i, fp in enumerate(morgan_fps) if fp is not None and maccs_list[i] is not None]
        if not valid_indices:
            continue
        morgan_fps = [morgan_fps[i] for i in valid_indices]
        X_morgan = np.vstack(morgan_fps).astype(float)
        
        # MACCS keys
        maccs_filtered = [maccs_list[i] for i in valid_indices]
        X_maccs = np.vstack(maccs_filtered).astype(float)
        
        # Continuous descriptors & fragment counts
        X_cont = features_df.iloc[valid_indices].values.astype(float)
        
        # Combine features
        X_combined = np.hstack([X_morgan, X_maccs, X_cont])
        y_combined = data_df[target].values[valid_indices]
        
        metrics = train_and_evaluate(X_combined, y_combined)
        metrics.update({'radius': radius, 'fp_size': size})
        combined_results.append(metrics)

combined_results_df = pd.DataFrame(combined_results)
combined_results_df.to_csv("combined_results.csv", index=False)
print("Combined features experiment results saved to combined_results.csv")


Morgan fingerprint experiment results saved to morgan_results.csv
MACCS experiment results saved to maccs_results.csv
Continuous descriptors/fragment counts experiment results saved to continuous_results.csv
Combined features experiment results saved to combined_results.csv
